In [1]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import OneHotEncoder
from scipy import stats
# from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from jupyter_dash import JupyterDash
from scipy import signal
from scipy.fft import fftshift
import plotly.graph_objects as go
import matplotlib.dates as mdates
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
import scipy.fftpack                 
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import butter, lfilter
from scipy.signal import find_peaks, peak_prominences
from scipy.signal import chirp, peak_widths
import math


In [3]:
from ahrs.filters import Madgwick, Mahony

## Histogram plots for raw data

In [5]:

def read_synced_files(folder_path):
    dfs = []
    file_list = os.listdir(folder_path)
    
    for filename in file_list:
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            df = pd.read_csv(file_path)
            dfs.append(df)
    dfs_concat = pd.concat(dfs)
    return dfs_concat
    
    # Now, 'dfs' is a list of DataFrames, each representing a CSV file in the folder


In [6]:
# Plots two histograms in one row for displaying hyperactive data and non-hyperactive datatypes Acceleration values.
def hyper_nonhpyer_plots(df,df2,filename):
    title_font = 20
    axis_font = 20
    bins=40
    font = {'family' : 'Sans'}
    plt.rc('font', **font)
    plt.rc('xtick', labelsize=10) 
    plt.rc('ytick', labelsize=10) 
    # rows,cols
    fig, axs  = plt.subplots(2, 2,figsize=(15, 5))
    plt.subplots_adjust(hspace = 0.5) # top=0.8-> for top title spacing.
    # First Row
    axs[0,0].set_title('Hyperactive (Accelerometer) \n' +filename, fontsize=title_font)
    axs[0,0].hist(df[' Accel_X'],bins=bins,color="red",label  = "X")
    axs[0,0].hist(df[' Accel_Y'],bins=bins,color="blue",label  = "Y")
    axs[0,0].hist(df[' Accel_Z'],bins=bins,color="green",label  = "Z")
    axs[0,1].set_title('Non Hyperactive (Accelerometer) \n' +filename, fontsize=title_font)
    axs[0,1].hist(df2[' Accel_X'],bins=bins,color="red",label  = "X")
    axs[0,1].hist(df2[' Accel_Y'],bins=bins,color="blue",label  = "Y")
    axs[0,1].hist(df2[' Accel_Z'],bins=bins,color="green",label  = "Z") 
    # Second Row
    axs[1,0].set_title('Hyperactive (Gryoscope) \n' +filename, fontsize=title_font)
    axs[1,0].hist(df[' Gyro_X'],bins=bins,color="red",label  = "X")
    axs[1,0].hist(df[' Gyro_Y'],bins=bins,color="blue",label  = "Y")
    axs[1,0].hist(df[' Gyro_Z'],bins=bins,color="green",label  = "Z")
    axs[1,1].set_title('Non Hyperactive (Gryoscope) \n' +filename, fontsize=title_font)
    axs[1,1].hist(df2[' Gyro_X'],bins=bins,color="red",label  = "X")
    axs[1,1].hist(df2[' Gyro_Y'],bins=bins,color="blue",label  = "Y")
    axs[1,1].hist(df2[' Gyro_Z'],bins=bins,color="green",label  = "Z") 
    # Add some space between subplots
    plt.tight_layout()
    plt.show()
               

In [7]:
def plot_behaviourtypes(df,filename):
    bh_type_unique = df["HyperactiveBehaviourType"].unique()
    # print("bh_type_unique: ",len(bh_type_unique),bh_type_unique)
    values_to_remove = ['-100', float('nan')]
    # Use NumPy's logical indexing to remove values
    bh_type_unique_filtered_array = bh_type_unique[~np.isin(bh_type_unique, values_to_remove)]
    # print("bh_type_unique NEW: ",len(bh_type_unique_filtered_array),bh_type_unique_filtered_array)
    title_font = 10
    axis_font = 20
    bins=40
    font = {'family' : 'Sans'}
    plt.rc('font', **font)
    plt.rc('xtick', labelsize=10) 
    plt.rc('ytick', labelsize=10) 
    #rows ,cols
    fig, axs  = plt.subplots(1, len(bh_type_unique_filtered_array),figsize=(15, 2))
    plt.subplots_adjust(hspace = 0.5) # top=0.8-> for top title spacing.
    for idx,val in enumerate(bh_type_unique_filtered_array):
        # print("idx,val: ",idx,val,type(val))
        title = val
        if val== "Manipulating object (turning pen over & over in hand)":
            title = "Manipulating object"
        # make a subset of behaviour type: df[df["HyperactiveBehaviourType"]==val]
        axs[idx].set_title(str(title) + ': Acc \n' +filename, fontsize=title_font)
        axs[idx].hist(df[df["HyperactiveBehaviourType"]==val][' Accel_X'],bins=bins,color="red",label  = "X")
        axs[idx].hist(df[df["HyperactiveBehaviourType"]==val][' Accel_Y'],bins=bins,color="blue",label  = "Y")
        axs[idx].hist(df[df["HyperactiveBehaviourType"]==val][' Accel_Z'],bins=bins,color="green",label  = "Z")
        axs[idx].legend()

     # Add some space between subplots
    plt.tight_layout()
    plt.show()

In [8]:
# using MATPLOT LIB we are able to print all the graphs on the screen itself.
def read_synced_files_and_plot_hist(folder_path):
    dfs = []
    file_list = os.listdir(folder_path)
    for filename in file_list:
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            print("filename: ",filename)
            print("===========================================================================================================================================")
            df = pd.read_csv(file_path,low_memory=False)
            df_hyperactive = df[df['Hyperactive_Restless'] == 1]
            df_no_hyperactive =df[df['Hyperactive_Restless'] == 0]
            hyper_nonhpyer_plots(df_hyperactive,df_no_hyperactive,filename)
            plot_behaviourtypes(df,filename)
   

In [ ]:
hist_folder = "/Users/shehjarsadhu/Desktop/UniversityOfRhodeIsland/Graduate/WBL/Project_Q2Behave/RenamedData/"
read_synced_files_and_plot_hist(hist_folder)
